# Theory in Code: Hybrid AI Patterns

Ten ways to couple a **language model** with **rules**, from *Hybrid AI Patterns* (Handschuh, 2026).

- One logic world for all patterns: `src/car_world.fx` (FrameX)
- One model call per step: NVIDIA Nemotron via OpenRouter (free)
- Only the **coupling** changes from pattern to pattern

**This notebook needs extra scripts and dependencies to run!**

## Setup

In [1]:
# FrameX binary is required on the system as well as the FrameX python bridge
from framex.framex import FrameXError

# llm.py is a helper script that defines ask and ask_json to connect to the model
from llm import ask, ask_json, FAST_MODEL, STRONG_MODEL

# world.py is a helper that builds the FrameX logic world
from world import car_world, reason_why_not, WORLD_FILE

print("fast model:  ", FAST_MODEL)
print("strong model:", STRONG_MODEL)

fast model:   nvidia/nemotron-3.5-lightning:free
strong model: nvidia/nemotron-3-super-120b-a12b:free


## Five invariants: true for every pattern

1. **Model text is data.** Nothing the model writes runs unchecked.
2. **Every model answer is a dated observation.** (`[LLM receipt]`)
3. **Rules decide.**
4. **Confidence is not truth.**
5. **Failures stay distinct:** failed call, wrong format, *unclear*, *unknown*.

## The car world

```
golf:Car[wheels -> 4; hasSteeringWheel -> true; brakes -> ok].
beetle:Car[wheels -> 4; hasSteeringWheel -> true; brakes -> broken].
tesla:Car[wheels -> 4; hasSteeringWheel -> true].        // brakes never checked

?C:Drivable    <- ?C:Car[wheels -> 4; hasSteeringWheel -> true; brakes -> ok].
?C:NotDrivable <- ?C:Car[brakes -> broken].
```

- `world closed.`: what cannot be derived is false
- golf: drivable · beetle: not drivable · tesla: **neither**, the rules cannot decide
- The file also holds speed limits (4), a checklist (5), diagnosis (8) and review rules (9)

<details>
<summary><b>Show the cars as a graph</b></summary>

<img src="images/car_world_cars.png" width="800">

</details>

In [ ]:
with car_world() as framex:
    for car in ["golf", "beetle", "tesla"]:
        drivable = framex.query(f"?- {car}:Drivable.")["status"]
        not_drivable = framex.query(f"?- {car}:NotDrivable.")["status"]
        print(f"{car:7} Drivable: {drivable:5}  NotDrivable: {not_drivable}")

golf    Drivable: true   NotDrivable: false
beetle  Drivable: false  NotDrivable: true
tesla   Drivable: false  NotDrivable: false


In [3]:
# The whole world, if you want to look at it
print(WORLD_FILE.read_text())

// car_world.fx
// The ONE small logic world that all ten pattern examples share.
// Every example loads this file unchanged and only adds the language model.

world closed.

// ---------------------------------------------------------------
// Schema: what a car is
// ---------------------------------------------------------------
Vehicle { wheels: Integer }.
Car extends Vehicle.
Car { hasSteeringWheel: Boolean }.

// ---------------------------------------------------------------
// Our three cars
// ---------------------------------------------------------------
golf:Car[wheels -> 4; hasSteeringWheel -> true; brakes -> ok].
beetle:Car[wheels -> 4; hasSteeringWheel -> true; brakes -> broken].
tesla:Car[wheels -> 4; hasSteeringWheel -> true].   // nobody has checked the brakes yet

// ---------------------------------------------------------------
// Can a car be driven?
// A car that is neither Drivable nor NotDrivable is a case
// the rules cannot decide (the tesla).
// ------------

---
## 1 · Model as Sensor

<img src="images/Patterns_1.png" width="300"> <img src="images/car_world_pattern_1.png" width="456">
  
**Model:** reads a workshop report, proposes facts, each with a verbatim quote.  
**Rules:** accept a reading only if its quote occurs in the text.  
**Watch:** tesla goes from `false` to `true`; `explain` shows the reading behind it.

**Known uses:**
- Assignment review: quote-checked extraction from contracts
- Campervan search: one classification per listing, from a fixed list or *unknown*
- Knowledge-graph construction by a language model (S. Pan et al. 2024)

In [4]:
# 1. Free text: the rules cannot read this, the model can.
text = "Workshop report: We checked the Tesla today. The brakes work fine."

# 2. The model acts as a sensor: it turns the text into proposed facts.
readings = ask_json(f"""Read this text about a car:
"{text}"

Return JSON with exactly these two keys:
  "brakes": "ok" or "broken"
  "wheels": a number
For each key give {{"value": ..., "quote": "<the exact words from the text>"}}.
If the text does not say it, use {{"value": "unknown", "quote": ""}}.
Return only the JSON.""")
print("The model proposes:", readings)

  [LLM receipt] 14:41:27  model=nvidia/nemotron-3.5-lightning:free
The model proposes: {'brakes': {'value': 'ok', 'quote': 'The brakes work fine.'}, 'wheels': {'value': 'unknown', 'quote': ''}}


In [5]:
with car_world() as framex:
    print("\nBefore: is the tesla drivable?", framex.query("?- tesla:Drivable.")["status"])

    # 3. The host only MEASURES: does the quote really occur in the text?
    for slot, reading in readings.items():
        value = str(reading["value"]).lower()
        if value == "unknown":
            continue  # the text does not say it, so there is nothing to propose
        quote_found = "true" if reading["quote"] and reading["quote"] in text else "false"
        framex.add(source=f"r_{slot}:Reading[car -> tesla; slot -> {slot}; "
                          f"value -> {value}; quoteFound -> {quote_found}].")

    # 4. The RULES decide which readings become facts (see car_world.fx).
    print("Accepted readings:", framex.query("?- ?R:Accepted.").get("bindings"))
    print("After:  is the tesla drivable?", framex.query("?- tesla:Drivable.")["status"])
    print("\nWhy?\n" + framex.explain("tesla:Drivable"))


Before: is the tesla drivable? false
Accepted readings: [{'R': 'r_brakes'}]
After:  is the tesla drivable? true

Why?
World assumption: closed
tesla:Drivable
  Derived by rule #0: ?C:Drivable <- ?C:Car AND ?C[wheels -> 4] AND ?C[hasSteeringWheel -> true] AND ?C[brakes -> ok].
  Bindings: C = tesla
  Because:
    tesla:Car
      Asserted fact
        Source: <json-source> | time: unspecified | weight: None | <json-source>:19:1
    tesla[wheels -> 4]
      Asserted fact
        Source: <json-source> | time: unspecified | weight: None | <json-source>:19:1
    tesla[hasSteeringWheel -> true]
      Asserted fact
        Source: <json-source> | time: unspecified | weight: None | <json-source>:19:1
    tesla[brakes -> ok]
      Derived by rule #4: ?C[?Slot -> ?Value] <- ?R:Accepted AND ?R[car -> ?C] AND ?R[slot -> ?Slot] AND ?R[value -> ?Value].
      Bindings: C = tesla, R = r_brakes, Slot = brakes, Value = ok
      Because:
        r_brakes:Accepted
          Derived by rule #3: ?R:Accepte

---
## 2 · Model as Author

<img src="images/Patterns_2.png" width="300">
  
**Model:** writes a `Roadworthy` rule from one policy sentence (stronger model).  
**Rules:** run test cases on the draft in a fresh copy of the world.  
**Watch:** only a person adopts; the checker only recommends.

**Known uses:**
- Rule drafting from one policy sentence, checked against hidden cases
- Decision-table drafts for EU Regulation 261/2004 (DMN round trip)
- Domain-model acquisition in LLM-Modulo (Kambhampati et al. 2024)

In [6]:
policy = "A car is roadworthy if it has 4 wheels and its brakes are ok."

# 1. The model drafts the rule. We give it a template, not a blank page.
draft = ask(f"""Translate this policy into ONE FrameX rule:
"{policy}"

FrameX example rule:
  ?X:Mortal <- ?X:Human[age -> ?A] AND ?A > 0.

Use exactly this head: ?X:Roadworthy <- ...
Available: class Car, slots wheels (a number) and brakes (ok or broken).
Return only the rule, ending with a dot.""", model=STRONG_MODEL).strip("` \n")
print("Draft rule:", draft)

  [LLM receipt] 14:41:51  model=nvidia/nemotron-3-super-120b-a12b:free
Draft rule: ?X:Roadworthy <- ?X:Car[wheels -> ?W] AND ?X:Car[brakes -> ?B] AND ?W = 4 AND ?B = ok.


In [7]:
# 2. Test cases, written BEFORE the draft: car -> expected answer.
tests = {"golf": "true", "beetle": "false", "tesla": "false"}

# 3. The checker: add the draft to a fresh copy of the world and run every test.
with car_world() as framex:
    try:
        framex.add(source=draft)
    except FrameXError as error:
        print("The draft does not parse:", error)
        raise SystemExit

    passed = 0
    for car, expected in tests.items():
        result = framex.query(f"?- {car}:Roadworthy.")["status"]
        passed += result == expected
        print(f"  {car:7} expected {expected:5}  got {result:5}  {'PASS' if result == expected else 'FAIL'}")

# 4. Only a person adopts a rule. The checker only recommends.
if passed == len(tests):
    print("\nAll tests pass: recommend the draft to a person for adoption.")
else:
    print(f"\nOnly {passed} of {len(tests)} tests pass: the draft is NOT adoptable.")

  golf    expected true   got true   PASS
  beetle  expected false  got false  PASS
  tesla   expected false  got false  PASS

All tests pass: recommend the draft to a person for adoption.


---
## 3 · Model as Translator

<img src="images/Patterns_3.png" width="300">
  
**Model:** question → query, and answer + proof → words.  
**Rules:** answer and prove.  
**Watch:** the host checks that the reply uses the rules' decision word.

**Known uses:**
- Assignment-review chat: the model calls the rules as a reviewer tool
- FrameX as a read-only server for model clients
- Logic-LM, LINC, SatLM: the model formalises, a solver answers

In [8]:
question = "Hey, can I take my Beetle for a drive today?"
print("Person asks:", question)

# 1. Words -> query. The model only gets the vocabulary, not the decision.
query = ask(f"""Translate the question into ONE FrameX query.
Known cars: golf, beetle, tesla. Known class: Drivable.
Example query: ?- golf:Drivable.
Question: "{question}"
Return only the query.""").strip("` \n")
print("Model's query:", query)

Person asks: Hey, can I take my Beetle for a drive today?
  [LLM receipt] 14:43:32  model=nvidia/nemotron-3.5-lightning:free
Model's query: ?- beetle:Drivable.


In [9]:
# 2. The rules answer, with a proof.
with car_world() as framex:
    verdict = framex.query(query)["status"]
    car = query.removeprefix("?-").split(":")[0].strip()
    if verdict == "true":
        proof = framex.explain(f"{car}:Drivable")
    else:  # the proof of the opposite, if the rules have one
        proof = framex.explain(f"{car}:NotDrivable")
print("Rules say:", verdict)

# 3. Answer + proof -> words for the person.
reply = ask(f"""A person asked: "{question}"
The rule engine answered: {verdict}
The proof: {proof}
Answer the person in one or two friendly sentences.
Start with YES or NO, matching the rule engine. Give only the reason from the proof.""")
print("\nModel replies:", reply)

# 4. The host checks that the reply uses the decision word of the rules.
expected_word = "YES" if verdict == "true" else "NO"
print("Reply matches the rules?", reply.upper().startswith(expected_word))

Rules say: false
  [LLM receipt] 14:44:08  model=nvidia/nemotron-3.5-lightning:free

Model replies: NO, because the beetle has broken brakes.
Reply matches the rules? True


---
## 4 · Model Proposes, Rules Dispose

<img src="images/Patterns_4.png" width="300"> <img src="images/car_world_pattern_4.png" width="445">
  
**Model:** proposes a speed for the golf near a school.  
**Rules:** admit it, or refuse with the failed condition (local limit: 30 km/h).  
**Watch:** the refusal reason goes back to the model; at most 3 attempts.

**Known uses:**
- FrameX-Agents: one action at a time, admitted against the task's capabilities
- Allocation recommendation with a materiality gate
- Guardrails (Rebedea et al. 2023); hard critics in LLM-Modulo

In [10]:
situation = "I am in my Golf and late for my exam! The road is empty. There is a school on the left."
feedback = ""

for attempt in range(1, 4):  # at most 3 proposals
    # 1. The model proposes.
    proposal = ask_json(f"""You help a driver. Situation: "{situation}"
{feedback}
Propose a driving speed. Return JSON: {{"speed": <km/h as a number>, "reason": "..."}}""")
    speed = proposal["speed"]
    print(f"Attempt {attempt}: model proposes {speed} km/h ({proposal['reason']})")

    # 2. The rules dispose (rule "Admitted" in car_world.fx).
    with car_world() as framex:
        framex.add(source=f"p:Proposal[car -> golf; speed -> {speed}; zone -> schoolZone].")
        admitted = framex.query("?- p:Admitted.")["status"] == "true"
        reason = reason_why_not(framex, "p:Admitted")

    # 3. The host executes only what the rules admit.
    if admitted:
        print(f"ADMITTED: the Golf drives at {speed} km/h.\n")
        break
    print(f"REFUSED: this condition is not met: {reason}\n")
    feedback = f"Your proposal of {speed} km/h was REFUSED by the rules. Failed condition: {reason}"
else:
    print("Stopped: no admissible proposal after 3 attempts.")

  [LLM receipt] 14:45:30  model=nvidia/nemotron-3.5-lightning:free
Attempt 1: model proposes 30 km/h (School zone ahead: reduced speed limits apply regardless of current traffic to protect children and pedestrians.)
ADMITTED: the Golf drives at 30 km/h.



---
## 5 · Rules Steer the Model

<img src="images/Patterns_5.png" width="300"> <img src="images/car_world_pattern_5.png" width="600">
  
**Model:** phrases the question and judges the answer (YES / NO / UNCLEAR).  
**Rules:** decide the next check and the end.  
**Watch:** interactive. Answer «hmm, maybe?» once, then «no» to one check.

**Known uses:**
- Interview, suitability and telephone-triage flows
- Customer support with hand-over to a person
- Programmable dialogue rails (Rebedea et al. 2023)

In [12]:
with car_world() as framex:
    while True:
        # 1. The rules decide the next step (rule "Next" in car_world.fx).
        step = framex.query("?- ?C:Next[topic -> ?T].")
        if step["status"] != "bindings":
            break  # no next check: the rules say we are done
        check, topic = step["bindings"][0]["C"], step["bindings"][0]["T"]

        # 2. The model only phrases the question.
        question = ask(f"Ask a driver one short, friendly yes/no question about: {topic}. "
                       "Return only the question.")
        answer = input(f"\n{question}\n> ")

        # 3. The model as judge: what does the answer mean?
        verdict = ask(f"""Question: "{question}"
Answer: "{answer}"
Does the answer mean yes or no? Reply with one word: YES, NO or UNCLEAR.""").upper()
        print(f"Judge: {verdict}")

        # 4. The rules decide what happens with the answer.
        if verdict == "YES":
            framex.add(source=f"{check}[answer -> yes].")
        elif verdict == "NO":
            framex.add(source=f"{check}[answer -> no].")
        else:
            print("The answer is unclear, so the rules ask the same check again.")

    # 5. The rules decide the outcome.
    failed = framex.query("?- ?C:Failed.")
    if failed["status"] == "bindings":
        print("\nDo NOT drive. Failed check:", failed["bindings"][0]["C"])
    else:
        print("\nAll checks passed. Drive safely!")

  [LLM receipt] 14:46:49  model=nvidia/nemotron-3.5-lightning:free
  [LLM receipt] 14:47:20  model=nvidia/nemotron-3.5-lightning:free
Judge: YES
  [LLM receipt] 14:47:31  model=nvidia/nemotron-3.5-lightning:free
  [LLM receipt] 14:47:44  model=nvidia/nemotron-3.5-lightning:free
Judge: YES
  [LLM receipt] 14:47:51  model=nvidia/nemotron-3.5-lightning:free
  [LLM receipt] 14:48:22  model=nvidia/nemotron-3.5-lightning:free
Judge: NO

Do NOT drive. Failed check: lightCheck


---
## 6 · Critique and Repair

<img src="images/Patterns_6.png" width="300">
  
**Model:** writes a FrameX fact for a bike.  
**Rules:** FrameX parses and checks it; the problem and the fix go back.  
**Watch:** each attempt fixes the error from the one before; stop on repetition.

**Known uses:**
- Format repair of dialogue turns and judges
- Rule-drafting loop driven by the checker's message
- Logic-LM refinement from solver errors; CRITIC (Gou et al. 2024)

In [13]:
task = "Add my bicycle: its name is bike, it is a Vehicle and it has 2 wheels."
prompt = f"Write one FrameX (F-Logic) fact for this: {task}\nReturn only the code."
feedback = ""
previous = None

for attempt in range(1, 4):
    code = ask(prompt + feedback).strip("` \n")
    print(f"Attempt {attempt}: {code}")

    if code == previous:
        print("Same answer as before: the repair is stuck. Stop.")
        break
    previous = code

    # 1. The checker classifies the problem.
    with car_world() as framex:
        try:
            framex.add(source=code)
            if framex.query("?- bike:Vehicle.")["status"] != "true":
                problem = "The fact does not make bike a Vehicle."
            elif framex.query("?- bike[wheels -> 2].")["status"] != "true":
                problem = "bike has no slot wheels with the number 2. Use exactly: wheels -> 2"
            else:
                problem = None
        except FrameXError as error:
            problem = f"Syntax error: {error.error['message']} (column {error.error['column']})."

    if problem is None:
        print("ACCEPTED: the fact passes every check.")
        break

    # 2. Name the problem AND the fix, then ask again.
    print(f"REJECTED: {problem}\n")
    feedback = (f"\n\nYour previous answer was:\n{code}\nIt was rejected: {problem}\n"
                "Hint: a FrameX fact looks like  object:Class[slot -> value; slot -> value].\n"
                "Slots are separated by ; and the fact ends with a dot.")
else:
    print("Stopped after 3 attempts.")

  [LLM receipt] 14:50:24  model=nvidia/nemotron-3.5-lightning:free
Attempt 1: bike : Vehicle . bike( name -> "bike", wheels -> 2 ).
REJECTED: Syntax error: expected ',' (column 29).

  [LLM receipt] 14:50:47  model=nvidia/nemotron-3.5-lightning:free
Attempt 2: bike:Vehicle[name -> "bike"; wheels -> 2].
ACCEPTED: the fact passes every check.


---
## 7 · Division by Competence

<img src="images/Patterns_7.png" width="300">
  
**Model:** assesses only the case the rules leave open (tesla).  
**Rules:** decide golf and beetle themselves.  
**Watch:** the model's answer is an assessment, not a fact; a person decides.

**Known uses:**
- Triage: *unclear* goes to a nurse
- FrameX-Agents: *awaiting user*
- Casework: required human review for evidence-gated actions

In [14]:
# What the owners told us (free text the rules cannot use).
notes = {
    "golf": "Runs great.",
    "beetle": "Brakes failed at the last inspection.",
    "tesla": "Yesterday the brakes made a loud grinding noise when I stopped.",
}

with car_world() as framex:
    for car, note in notes.items():
        # 1. The rules decide what they can.
        if framex.query(f"?- {car}:Drivable.")["status"] == "true":
            print(f"{car:7} RULES decide: drivable")
        elif framex.query(f"?- {car}:NotDrivable.")["status"] == "true":
            print(f"{car:7} RULES decide: not drivable")
        else:
            # 2. The rules cannot decide: hand the case to the model.
            assessment = ask_json(f"""The owner of a car says: "{note}"
Should this car be driven? Return JSON:
{{"assessment": "probably drivable" or "probably not drivable" or "unclear", "reason": "..."}}""")
            print(f"{car:7} rules cannot decide -> MODEL assesses: "
                  f"{assessment['assessment']} ({assessment['reason']})")
            # 3. The assessment is not a fact. A person decides.
            print(f"{'':7} -> sent to a mechanic for the final decision.")

golf    RULES decide: drivable
beetle  RULES decide: not drivable
  [LLM receipt] 14:52:29  model=nvidia/nemotron-3.5-lightning:free
tesla   rules cannot decide -> MODEL assesses: probably not drivable (A loud grinding noise during braking typically indicates severely worn brake pads or metal-on-metal contact, which can compromise stopping power and cause further damage. For safety, the vehicle should not be driven until inspected and repaired by a qualified mechanic.)
        -> sent to a mechanic for the final decision.


---
## 8 · Model as Hypothesis Generator

<img src="images/Patterns_8.png" width="300"> <img src="images/car_world_pattern_8.png" width="560">
  
**Model:** proposes causes for «my Beetle does not start» (stronger model).  
**Rules:** test each cause in a fresh copy: does it explain, does it contradict?  
**Watch:** survivors get the question that decides; no guess becomes a fact.

**Known uses:**
- Bank reconciliation: the model proposes causes, rules test them
- Evidence-needs search over open questions
- Abduction tested against retrieved events (Shi et al. 2023)

In [15]:
observation = "My Beetle does not start."

# 1. The model proposes possible causes.
hypotheses = ask_json(f"""A driver says: "{observation}"
Propose the 4 most likely causes.
Describe each cause with a slot and a value, for example {{"slot": "battery", "value": "empty"}}.
Slots the rules know: battery, fuel, starter, immobilizer.
Values the rules know: empty, broken, active.
Return JSON: [{{"slot": "...", "value": "..."}}, ...]""", model=STRONG_MODEL)

# print(hypotheses)

  [LLM receipt] 14:54:56  model=nvidia/nemotron-3-super-120b-a12b:free


In [16]:
# 2. The rules test every hypothesis in a fresh copy of the world.
for h in hypotheses:
    guess = f"beetle[{h['slot']} -> {h['value']}]"
    with car_world() as framex:
        try:
            framex.add(source=guess + ".")  # assumed only inside this copy
        except FrameXError:
            print(f"{guess:35} DROPPED: not valid FrameX, the rules cannot test it")
            continue
        explains = framex.query("?- beetle:WontStart.")["status"] == "true"
        contradicts = framex.query("?- beetle:Contradiction.")["status"] == "true"
        question = framex.query("?- beetle[askNext -> ?Q].").get("bindings", [{}])[0].get("Q")

    if not explains:
        print(f"{guess:35} DROPPED: the rules cannot explain the observation with it")
    elif contradicts:
        print(f"{guess:35} DROPPED: contradicts what we see (fuel gauge full, lights on)")
    else:
        print(f"{guess:35} POSSIBLE -> ask: {question}")

beetle[battery -> empty]            DROPPED: contradicts what we see (fuel gauge full, lights on)
beetle[fuel -> empty]               DROPPED: contradicts what we see (fuel gauge full, lights on)
beetle[starter -> broken]           POSSIBLE -> ask: "Do you hear a click when you turn the key?"
beetle[immobilizer -> active]       POSSIBLE -> ask: "Does the key symbol blink on the dashboard?"


---
## 9 · Model as Critic of the Rules

<img src="images/Patterns_9.png" width="300"> <img src="images/car_world_pattern_9.png" width="339">
  
**Model:** reviews the `Drivable` rule against the policy text (stronger model).  
**Rules:** keep a remark only if it quotes the policy exactly.  
**Watch:** findings go to a person; the rules stay unchanged.

**Known uses:**
- Policy critic in a university-administration case (built, not yet measured)
- Soft critics in LLM-Modulo (Kambhampati et al. 2024)
- Cross-reflection (Liu et al. 2025)

In [17]:
policy = ("A car may be driven if it has 4 wheels, a steering wheel and working brakes. "
          "It also needs a valid inspection sticker.")

# The rules under review: the "Drivable" rule from car_world.fx.
rules = [line for line in WORLD_FILE.read_text().splitlines() if line.startswith("?C:Drivable")]
print("Rule under review:", rules[0])

# 1. The model criticises the rules.
remarks = ask_json(f"""Policy text: "{policy}"
FrameX rule: {rules[0]}

Does the rule implement the policy correctly? List every problem.
Return JSON: [{{"quote": "<exact words from the policy>", "remark": "..."}}]
Return [] if the rule is correct.""", model=STRONG_MODEL)

# print(remarks)

Rule under review: ?C:Drivable    <- ?C:Car[wheels -> 4; hasSteeringWheel -> true; brakes -> ok].
  [LLM receipt] 14:56:24  model=nvidia/nemotron-3-super-120b-a12b:free


In [18]:
with car_world() as framex:
    # 2. The host measures: is the quote really in the policy?
    for i, r in enumerate(remarks):
        found = "true" if r["quote"] in policy else "false"
        framex.add(source=f"remark{i}:Remark[quoteFound -> {found}].")

    # 3. The rules decide which remarks become findings for a person.
    for i, r in enumerate(remarks):
        is_finding = framex.query(f"?- remark{i}:Finding.")["status"] == "true"
        label = "FINDING for the reviewer" if is_finding else "DROPPED (quote not in policy)"
        print(f"\n{label}\n  quote:  {r['quote']}\n  remark: {r['remark']}")

    # 4. The rules themselves are unchanged.
    print("\nRules unchanged. Is the golf still drivable?",
          framex.query("?- golf:Drivable.")["status"])


FINDING for the reviewer
  quote:  It also needs a valid inspection sticker.
  remark: The rule does not include a condition for a valid inspection sticker, so it does not fully implement the policy.

Rules unchanged. Is the golf still drivable? true


---
## 10 · Rules Teach the Model

<img src="images/Patterns_10.png" width="300">
  
**Rules:** label example cars, with the reason from the proof.  
**Model:** answers new cars with the labelled examples in its prompt.  
**Watch:** the rules grade every answer; they stay the check.

**Known uses:**
- Labels, examples and grading for 144 generated admission cases
- Kautz's Neuro:Symbolic→Neuro: rules generate training data
- Constitutional AI and RLAIF (Bai et al. 2022)

In [19]:
examples = {
    "carA": "wheels -> 4; hasSteeringWheel -> true; brakes -> ok",
    "carB": "wheels -> 4; hasSteeringWheel -> true; brakes -> broken",
    "carC": "wheels -> 3; hasSteeringWheel -> true; brakes -> ok",
    "carD": "wheels -> 4; hasSteeringWheel -> false; brakes -> ok",
}
new_cases = {
    "carE": "wheels -> 6; hasSteeringWheel -> true; brakes -> ok",
    "carF": "wheels -> 4; hasSteeringWheel -> true; brakes -> ok",
    "carG": "wheels -> 4; hasSteeringWheel -> false; brakes -> broken",
}

with car_world() as framex:
    for car, slots in {**examples, **new_cases}.items():
        framex.add(source=f"{car}:Car[{slots}].")

    def label(car):
        return framex.query(f"?- {car}:Drivable.")["status"]

    # 1. The rules label the examples, with the reason from the proof.
    lesson = ""
    for car, slots in examples.items():
        if label(car) == "true":
            reason = "all conditions met"
        else:
            reason = f"missing: {reason_why_not(framex, f'{car}:Drivable')}"
        lesson += f"Car [{slots}] -> drivable: {label(car)} ({reason})\n"
    print("Lesson written by the rules:\n" + lesson)

    # 2. The model answers new cases, with the lesson in its prompt.
    # 3. The rules grade every answer.
    score = 0
    for car, slots in new_cases.items():
        answer = ask(f"""Examples labelled by our rules:
{lesson}
Car [{slots}] -> drivable? Answer only true or false.""").lower()
        correct = label(car)
        score += answer == correct
        print(f"{car}: model says {answer:5}  rules say {correct:5}  {'RIGHT' if answer == correct else 'WRONG'}")
    print(f"\nScore: {score} of {len(new_cases)}")

Lesson written by the rules:
Car [wheels -> 4; hasSteeringWheel -> true; brakes -> ok] -> drivable: true (all conditions met)
Car [wheels -> 4; hasSteeringWheel -> true; brakes -> broken] -> drivable: false (missing: carB[brakes -> ok])
Car [wheels -> 3; hasSteeringWheel -> true; brakes -> ok] -> drivable: false (missing: carC[wheels -> 4])
Car [wheels -> 4; hasSteeringWheel -> false; brakes -> ok] -> drivable: false (missing: carD[hasSteeringWheel -> true])

  [LLM receipt] 14:58:10  model=nvidia/nemotron-3.5-lightning:free
carE: model says false  rules say false  RIGHT
  [LLM receipt] 14:58:29  model=nvidia/nemotron-3.5-lightning:free
carF: model says true   rules say true   RIGHT
  [LLM receipt] 14:58:41  model=nvidia/nemotron-3.5-lightning:free
carG: model says false  rules say false  RIGHT

Score: 3 of 3


---
## Which pattern when?

| Problem | Pattern |
|---|---|
| Facts are in text | 1 Sensor |
| A policy must become a rule | 2 Author |
| People ask in their own words | 3 Translator |
| The model acts | 4 Proposes, Dispose |
| A long process or conversation | 5 Steer |
| A reply fails a check | 6 Repair |
| Some cases are ambiguous | 7 Competence |
| Something must be explained | 8 Hypotheses |
| Rule changes with incomplete checks | 9 Critic |
| The model must learn the domain | 10 Teach |